# Healthcare Data Cleaning
Create, clean, and verify the patient dataset per the task requirements.

Create the DataFrame using the provided code.

In [ ]:
!pip install numpy pandas
import pandas as pd
import numpy as np
data = {
    'patient_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110,
                   111, 112, 113, 114, 115, 101, 107, 118, 119, 120],
    'age': ['25', '34', None, '45', '29', None, '38', '52', '27', '41',
            '33', 'unknown', '48', '26', '35', '25', '38', '31', None, '44'],
    'weight': ['70', '65', '80', None, '75', None, '68', '90', '72', '85',
               '78', None, '82', '69', 'N/A', '70', '68', '74', None, '88'],
    'blood_pressure': [120, 130, None, 140, 125, None, 135, None, 118, 145,
                      128, None, 138, 122, None, 120, 135, 126, None, 142],
    'medication': ['Aspirin', 'Metformin', 'Lisinopril', None, 'Aspirin',
                   'Metformin', 'Lisinopril', 'Aspirin', None, 'Metformin',
                   'Lisinopril', 'Aspirin', None, 'Metformin', 'Aspirin',
                   'Aspirin', 'Lisinopril', 'Metformin', 'Aspirin', None],
    'insurance_provider': ['Blue Cross', 'Aetna', 'Cigna', 'UnitedHealth', None,
                          'Blue Cross', 'Aetna', 'Cigna', 'UnitedHealth', 'Blue Cross',
                          'Aetna', None, 'UnitedHealth', 'Blue Cross', 'Aetna',
                          'Blue Cross', 'Aetna', 'Cigna', 'UnitedHealth', None]
}
df = pd.DataFrame(data)
df.head()

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


,patient_id,age,weight,blood_pressure,medication,insurance_provider
0,101,25,70,120.0,Aspirin,Blue Cross
1,102,34,65,130.0,Metformin,Aetna
2,103,NaN,80,NaN,Lisinopril,Cigna
3,104,45,NaN,140.0,NaN,UnitedHealth
4,105,29,75,125.0,Aspirin,NaN


: 

Task 1: Inspect the data (info, missing counts, missing percentage, duplicates).

In [3]:
print('--- df.info() ---')
df.info()
print('\n--- Missing value counts ---')
print(df.isnull().sum())
print('\n--- Missing value percentage ---')
print((df.isnull().sum() / len(df)) * 100)
print('\n--- Duplicate rows (count) ---')
print(df.duplicated().sum())

--- df.info() ---
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   patient_id          20 non-null     int64  
 1   age                 17 non-null     str    
 2   weight              16 non-null     str    
 3   blood_pressure      14 non-null     float64
 4   medication          16 non-null     str    
 5   insurance_provider  17 non-null     str    
dtypes: float64(1), int64(1), str(4)
memory usage: 1.1 KB

--- Missing value counts ---
patient_id            0
age                   3
weight                4
blood_pressure        6
medication            4
insurance_provider    3
dtype: int64

--- Missing value percentage ---
patient_id             0.0
age                   15.0
weight                20.0
blood_pressure        30.0
medication            20.0
insurance_provider    15.0
dtype: float64

--- Duplicate rows (count) ---
2


Task 2: Data type conversions (convert `age` and `weight` to numeric; `insurance_provider` to category).

In [4]:
# Record missing counts before conversions
missing_before = df.isnull().sum()
# Convert age and weight to numeric, coercing errors to NaN
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['weight'] = pd.to_numeric(df['weight'], errors='coerce')
# Count new missing values created by conversion
missing_after_conversion = df.isnull().sum()
print('Missing values before conversions:\n', missing_before)
print('\nMissing values after conversions:\n', missing_after_conversion)
print('\nNew missing values created by conversions:\n', missing_after_conversion - missing_before)
# Convert insurance_provider to category
df['insurance_provider'] = df['insurance_provider'].astype('category')
print('\nData types after conversions:\n', df.dtypes)

Missing values before conversions:
 patient_id            0
age                   3
weight                4
blood_pressure        6
medication            4
insurance_provider    3
dtype: int64

Missing values after conversions:
 patient_id            0
age                   4
weight                5
blood_pressure        6
medication            4
insurance_provider    3
dtype: int64

New missing values created by conversions:
 patient_id            0
age                   1
weight                1
blood_pressure        0
medication            0
insurance_provider    0
dtype: int64

Data types after conversions:
 patient_id               int64
age                    float64
weight                 float64
blood_pressure         float64
medication                 str
insurance_provider    category
dtype: object


Task 3: Handle missing values (fill age, weight, blood_pressure with median; medication with mode; insurance_provider with 'Unknown').

In [5]:
# Fill numeric columns with medians where appropriate
df['age'] = df['age'].fillna(df['age'].median())
df['weight'] = df['weight'].fillna(df['weight'].median())
df['blood_pressure'] = df['blood_pressure'].fillna(df['blood_pressure'].median())
# Fill medication with mode
df['medication'] = df['medication'].fillna(df['medication'].mode()[0])
# Fill insurance_provider with constant 'Unknown' and keep as category
df['insurance_provider'] = df['insurance_provider'].cat.add_categories(['Unknown'])
df['insurance_provider'] = df['insurance_provider'].fillna('Unknown')
print('Missing values after filling:')
print(df.isnull().sum())

Missing values after filling:
patient_id            0
age                   0
weight                0
blood_pressure        0
medication            0
insurance_provider    0
dtype: int64


Task 4: Handle duplicates — show duplicates, identify by `patient_id`, and remove duplicates keeping first occurrence.

In [6]:
print('Shape before removing duplicates:', df.shape)
print('\nDuplicate rows (all columns) where keep=False:')
display(df[df.duplicated(keep=False)])
print('\nDuplicates based on patient_id (boolean mask):')
print(df.duplicated(subset=['patient_id']))
# Remove duplicates based on patient_id, keep first
df = df.drop_duplicates(subset=['patient_id'], keep='first').reset_index(drop=True)
print('\nShape after removing duplicates:', df.shape)

Shape before removing duplicates: (20, 6)

Duplicate rows (all columns) where keep=False:


,patient_id,age,weight,blood_pressure,medication,insurance_provider
0,101,25.0,70.0,120.0,Aspirin,Blue Cross
6,107,38.0,68.0,135.0,Lisinopril,Aetna
15,101,25.0,70.0,120.0,Aspirin,Blue Cross
16,107,38.0,68.0,135.0,Lisinopril,Aetna



Duplicates based on patient_id (boolean mask):
0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15     True
16     True
17    False
18    False
19    False
dtype: bool

Shape after removing duplicates: (18, 6)


Task 5: Complete workflow verification — reload original DataFrame, create `df_clean`, perform all steps in order, and produce a verification report.

In [7]:
# Reload original data to start fresh
df_orig = pd.DataFrame(data)
df_clean = df_orig.copy()
# Report initial state
report = {}
report['shape_before'] = df_clean.shape
report['missing_before'] = df_clean.isnull().sum()
report['duplicates_before'] = df_clean.duplicated().sum()
report['dtypes_before'] = df_clean.dtypes
# 1) types
df_clean['age'] = pd.to_numeric(df_clean['age'], errors='coerce')
df_clean['weight'] = pd.to_numeric(df_clean['weight'], errors='coerce')
df_clean['insurance_provider'] = df_clean['insurance_provider'].astype('category')
# 2) missing values
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())
df_clean['weight'] = df_clean['weight'].fillna(df_clean['weight'].median())
df_clean['blood_pressure'] = df_clean['blood_pressure'].fillna(df_clean['blood_pressure'].median())
df_clean['medication'] = df_clean['medication'].fillna(df_clean['medication'].mode()[0])
df_clean['insurance_provider'] = df_clean['insurance_provider'].cat.add_categories(['Unknown'])
df_clean['insurance_provider'] = df_clean['insurance_provider'].fillna('Unknown')
# 3) duplicates based on patient_id
report['duplicates_before_patient_id'] = df_clean.duplicated(subset=['patient_id']).sum()
df_clean = df_clean.drop_duplicates(subset=['patient_id'], keep='first').reset_index(drop=True)
# Final report values
report['shape_after'] = df_clean.shape
report['missing_after'] = df_clean.isnull().sum()
report['duplicates_after'] = df_clean.duplicated().sum()
report['dtypes_after'] = df_clean.dtypes
# Display verification report
print('--- Verification Report ---')
print('Shape before -> after:', report['shape_before'], '->', report['shape_after'])
print('\nMissing before -> after:\n', report['missing_before'], '\n->\n', report['missing_after'])
print('\nDuplicates before -> after:', report['duplicates_before'], '->', report['duplicates_after'])
print('\nDuplicates based on patient_id before removal:', report['duplicates_before_patient_id'])
print('\nData types before:\n', report['dtypes_before'])
print('\nData types after:\n', report['dtypes_after'])
# Show cleaned DataFrame head
display(df_clean.head())

--- Verification Report ---
Shape before -> after: (20, 6) -> (18, 6)

Missing before -> after:
 patient_id            0
age                   3
weight                4
blood_pressure        6
medication            4
insurance_provider    3
dtype: int64 
->
 patient_id            0
age                   0
weight                0
blood_pressure        0
medication            0
insurance_provider    0
dtype: int64

Duplicates before -> after: 2 -> 0

Duplicates based on patient_id before removal: 2

Data types before:
 patient_id              int64
age                       str
weight                    str
blood_pressure        float64
medication                str
insurance_provider        str
dtype: object

Data types after:
 patient_id               int64
age                    float64
weight                 float64
blood_pressure         float64
medication                 str
insurance_provider    category
dtype: object


,patient_id,age,weight,blood_pressure,medication,insurance_provider
0,101,25.0,70.0,120.0,Aspirin,Blue Cross
1,102,34.0,65.0,130.0,Metformin,Aetna
2,103,34.5,80.0,129.0,Lisinopril,Cigna
3,104,45.0,74.0,140.0,Aspirin,UnitedHealth
4,105,29.0,75.0,125.0,Aspirin,Unknown
